# ASI Assignment — Regression (Student A)

Paper: *Simple and Scalable Predictive Uncertainty Estimation using Deep Ensembles* (Lakshminarayanan et al., NeurIPS 2017)

**Group settings:** PyTorch, ensemble size $M=5$, adversarial perturbation $\varepsilon = 0.01 \times$ (training range per input dimension).

This notebook covers:
1. Heteroscedastic Gaussian NLL (Eq. 1)
2. 1D toy regression → reproduce Figure 1
3. Boston Housing benchmark → RMSE & NLL (single net vs. deep ensemble)

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import fetch_openml

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'PyTorch: {torch.__version__}')

M_ENSEMBLE = 5
NUM_EPOCHS = 40          # Boston / UCI benchmarks (paper Sec. 3.3)
LR = 1e-3

# Toy needs more steps: y = x^3 spans ~[-60, 60] with only 20 points
NUM_EPOCHS_TOY = 2000
LR_TOY = 0.01

MIN_VAR = 1e-6

## Eq. (1): Heteroscedastic Gaussian NLL

Assume $y \mid x \sim \mathcal{N}(\mu_\theta(x), \sigma^2_\theta(x))$. The log-likelihood is

$$\log p_\theta(y \mid x) = -\tfrac{1}{2}\log(2\pi\sigma^2_\theta) - \tfrac{(y-\mu_\theta(x))^2}{2\sigma^2_\theta}.$$

Minimizing **negative** log-likelihood (ignoring constants) gives the training loss implemented below:

$$\mathcal{L}_{\mathrm{NLL}} = \tfrac{1}{2}\log\sigma^2_\theta + \tfrac{(y-\mu_\theta)^2}{2\sigma^2_\theta}.$$

The network outputs $\mu$ and a raw variance head passed through `softplus` for $\sigma^2 > 0$ (plus `MIN_VAR` for stability), as in the paper.

In [ ]:
def gaussian_nll(y, mu, var):
    """Per-sample Gaussian NLL; y, mu, var shape (N, 1)."""
    return 0.5 * (torch.log(var) + (y - mu).pow(2) / var).mean()


def gaussian_nll_per_sample(y, mu, var):
    return 0.5 * (torch.log(var) + (y - mu).pow(2) / var)


class RegressionMLP(nn.Module):
    """1-hidden-layer MLP. heteroscedastic=True → (mu, var); else → point prediction."""

    def __init__(self, input_dim=1, hidden_dim=100, heteroscedastic=True):
        super().__init__()
        self.heteroscedastic = heteroscedastic
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        out_dim = 2 if heteroscedastic else 1
        self.fc2 = nn.Linear(hidden_dim, out_dim)

    def forward(self, x):
        h = F.relu(self.fc1(x))
        out = self.fc2(h)
        if self.heteroscedastic:
            mu = out[:, :1]
            var = F.softplus(out[:, 1:]) + MIN_VAR
            return mu, var
        return out


def predict_gaussian(model, x):
    model.eval()
    with torch.no_grad():
        mu, var = model(x)
    return mu, var


def ensemble_aggregate_gaussian(models, x):
    """Moment-matched Gaussian mixture (paper Sec. 2.4)."""
    mus, vars_ = [], []
    for m in models:
        mu, var = predict_gaussian(m, x)
        mus.append(mu)
        vars_.append(var)
    mus = torch.stack(mus, dim=0)      # (M, N, 1)
    vars_ = torch.stack(vars_, dim=0)
    mu_star = mus.mean(dim=0)
    var_star = (vars_ + mus.pow(2)).mean(dim=0) - mu_star.pow(2)
    var_star = torch.clamp(var_star, min=MIN_VAR)
    return mu_star, var_star


def ensemble_aggregate_mse(models, x):
    """MSE-trained nets: mean prediction + empirical variance across nets."""
    preds = []
    for m in models:
        m.eval()
        with torch.no_grad():
            preds.append(m(x))
    preds = torch.stack(preds, dim=0)
    mu_star = preds.mean(dim=0)
    var_star = preds.var(dim=0, unbiased=False) + MIN_VAR
    return mu_star, var_star


def compute_epsilon_per_dim(x_train_np):
    """ε = 0.01 × (max - min) per feature (training set)."""
    ranges = x_train_np.max(axis=0) - x_train_np.min(axis=0)
    return 0.01 * np.maximum(ranges, 1e-8)


def train_regression_model(
    x_train, y_train,
    *,
    heteroscedastic=True,
    use_adversarial=False,
    epsilon_per_dim=None,
    hidden_dim=100,
    seed=0,
    verbose=False,
    lr=LR,
    num_epochs=NUM_EPOCHS,
):
    torch.manual_seed(seed)
    np.random.seed(seed)

    model = RegressionMLP(
        input_dim=x_train.shape[1],
        hidden_dim=hidden_dim,
        heteroscedastic=heteroscedastic,
    ).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    if epsilon_per_dim is None:
        epsilon_per_dim = compute_epsilon_per_dim(x_train.detach().cpu().numpy())
    eps = torch.tensor(epsilon_per_dim, dtype=x_train.dtype, device=device).view(1, -1)

    for epoch in range(num_epochs):
        model.train()
        optimizer.zero_grad()

        if use_adversarial and heteroscedastic:
            x_in = x_train.detach().clone().requires_grad_(True)
            mu, var = model(x_in)
            loss = gaussian_nll(y_train, mu, var)
            loss.backward(retain_graph=True)
            perturb = eps * x_in.grad.sign()
            x_adv = (x_in + perturb).detach()
            mu_adv, var_adv = model(x_adv)
            loss_adv = gaussian_nll(y_train, mu_adv, var_adv)
            (loss + loss_adv).backward()
        elif heteroscedastic:
            mu, var = model(x_train)
            loss = gaussian_nll(y_train, mu, var)
            loss.backward()
        else:
            mu = model(x_train)
            loss = F.mse_loss(mu, y_train)
            loss.backward()

        optimizer.step()

        if verbose and (epoch + 1) % 10 == 0:
            print(f'  epoch {epoch+1}/{num_epochs}, loss={loss.item():.4f}')

    return model


def train_ensemble(x_train, y_train, n_models=M_ENSEMBLE, **kwargs):
    models = []
    for m in range(n_models):
        models.append(train_regression_model(x_train, y_train, seed=m, **kwargs))
    return models


def evaluate_rmse_nll(y_true, mu, var):
    mu = torch.as_tensor(mu, dtype=torch.float32, device=device)
    var = torch.as_tensor(var, dtype=torch.float32, device=device).view_as(mu)
    y_true = torch.as_tensor(y_true, dtype=torch.float32, device=device).view_as(mu)
    rmse = torch.sqrt(((y_true - mu) ** 2).mean()).item()
    nll = gaussian_nll_per_sample(y_true, mu, var).mean().item()
    return rmse, nll


def plot_uncertainty_band(ax, x, mu, std, x_train, y_train, title, color='gray'):
  x_np = x.cpu().numpy().ravel()
  mu_np = mu.cpu().numpy().ravel()
  std_np = std.cpu().numpy().ravel()
  xt = x_train.cpu().numpy().ravel()
  yt = y_train.cpu().numpy().ravel()
  order = np.argsort(x_np)
  ax.plot(x_np, x_np ** 3, 'b-', lw=2, label='ground truth $x^3$')
  ax.scatter(xt, yt, c='red', s=30, zorder=5, label='train data')
  ax.plot(x_np[order], mu_np[order], color=color, lw=1.5, label='predicted mean')
  ax.fill_between(
      x_np[order],
      mu_np[order] - 3 * std_np[order],
      mu_np[order] + 3 * std_np[order],
      color=color, alpha=0.25, label='$\pm 3\sigma$',
  )
  ax.set_title(title)
  ax.set_xlabel('x')
  ax.set_ylabel('y')
  ax.set_xlim(-6, 6)
  ax.legend(loc='upper left', fontsize=8)

## 1D Toy Dataset (Figure 1)

Following the paper (Sec. 3.2): 20 training points with $y = x^3 + \epsilon$, $\epsilon \sim \mathcal{N}(0, 3^2)$, $x \sim \mathrm{Uniform}[-4, 4]$.

**Note:** With only 20 points and $|y|$ up to $\sim 60$, use `LR_TOY=0.01` and `NUM_EPOCHS_TOY=2000`. Using `lr=1e-3` and 40 epochs leaves the predicted mean flat (underfitting).

In [ ]:
def make_toy_data(n_train=20, seed=42):
    rng = np.random.RandomState(seed)
    x_train = rng.uniform(-4, 4, size=(n_train, 1)).astype(np.float32)
    y_train = (x_train ** 3 + rng.normal(0, 3, size=(n_train, 1))).astype(np.float32)
    x_test = np.linspace(-6, 6, 400, dtype=np.float32).reshape(-1, 1)
    return (
        torch.tensor(x_train, device=device),
        torch.tensor(y_train, device=device),
        torch.tensor(x_test, device=device),
    )


x_tr, y_tr, x_te = make_toy_data()
eps_toy = compute_epsilon_per_dim(x_tr.cpu().numpy())
print(f'Toy ε (1D): {eps_toy[0]:.4f}  (0.01 × range = 0.01 × 8)')

print(f'Training Figure 1 models (lr={LR_TOY}, epochs={NUM_EPOCHS_TOY})...')
toy_kw = dict(lr=LR_TOY, num_epochs=NUM_EPOCHS_TOY, hidden_dim=100)
mse_ensemble = train_ensemble(x_tr, y_tr, heteroscedastic=False, **toy_kw)
nll_single = train_regression_model(
    x_tr, y_tr, heteroscedastic=True, use_adversarial=False, seed=0, **toy_kw
)
nll_at_single = train_regression_model(
    x_tr, y_tr, heteroscedastic=True, use_adversarial=True,
    epsilon_per_dim=eps_toy, seed=0, **toy_kw
)
nll_ensemble = train_ensemble(
    x_tr, y_tr, heteroscedastic=True, use_adversarial=False, **toy_kw
)

fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharey=True)

mu, var = ensemble_aggregate_mse(mse_ensemble, x_te)
plot_uncertainty_band(axes[0], x_te, mu, torch.sqrt(var), x_tr, y_tr,
                      'MSE ensemble ($M=5$), empirical variance')

mu, var = predict_gaussian(nll_single, x_te)
plot_uncertainty_band(axes[1], x_te, mu, torch.sqrt(var), x_tr, y_tr,
                      'NLL, single network')

mu, var = predict_gaussian(nll_at_single, x_te)
plot_uncertainty_band(axes[2], x_te, mu, torch.sqrt(var), x_tr, y_tr,
                      'NLL + adversarial, single network')

mu, var = ensemble_aggregate_gaussian(nll_ensemble, x_te)
plot_uncertainty_band(axes[3], x_te, mu, torch.sqrt(var), x_tr, y_tr,
                      f'NLL deep ensemble ($M={M_ENSEMBLE}$)')

plt.suptitle('Toy 1D regression — reproduction of Figure 1', y=1.02)
plt.tight_layout()
plt.savefig('toy_figure1_reproduction.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: toy_figure1_reproduction.png')

## Boston Housing Benchmark

Protocol from Hernandez-Lobato & Adams [24] / paper Sec. 3.3: 1 hidden layer, 50 units, 40 epochs, $M=5$. We use 5-fold CV (paper uses 20 folds; reduce for runtime in class project).

In [ ]:
def load_boston_xy():
    boston = fetch_openml(data_id=531, as_frame=True, parser='auto')
    X = boston.data.astype(np.float32).values
    y = boston.target.astype(np.float32).values.reshape(-1, 1)
    return X, y


def run_boston_cv(n_splits=5, n_models=M_ENSEMBLE):
    X, y = load_boston_xy()
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=0)

    rows = {'single_nll': [], 'ensemble_nll': [], 'single_rmse': [], 'ensemble_rmse': []}

    for fold, (tr_idx, te_idx) in enumerate(kf.split(X)):
        print(f'Fold {fold + 1}/{n_splits}')
        X_tr, X_te = X[tr_idx], X[te_idx]
        y_tr, y_te = y[tr_idx], y[te_idx]

        x_scaler = StandardScaler().fit(X_tr)
        y_scaler = StandardScaler().fit(y_tr)
        X_tr_s = x_scaler.transform(X_tr)
        X_te_s = x_scaler.transform(X_te)
        y_tr_s = y_scaler.transform(y_tr)
        y_te_s = y_scaler.transform(y_te)

        x_tr_t = torch.tensor(X_tr_s, device=device)
        y_tr_t = torch.tensor(y_tr_s, device=device)
        x_te_t = torch.tensor(X_te_s, device=device)
        y_te_t = torch.tensor(y_te_s, device=device)

        eps = compute_epsilon_per_dim(X_tr_s)

        single = train_regression_model(
            x_tr_t, y_tr_t, heteroscedastic=True, hidden_dim=50, seed=0, epsilon_per_dim=eps
        )
        ensemble = train_ensemble(
            x_tr_t, y_tr_t, n_models=n_models, heteroscedastic=True, hidden_dim=50, epsilon_per_dim=eps
        )

        mu_s, var_s = predict_gaussian(single, x_te_t)
        mu_e, var_e = ensemble_aggregate_gaussian(ensemble, x_te_t)

        # RMSE on original y scale; NLL on standardized scale
        y_std_scale = float(y_scaler.scale_[0])
        mu_s_orig = torch.tensor(y_scaler.inverse_transform(mu_s.cpu().numpy()), dtype=torch.float32, device=device)
        mu_e_orig = torch.tensor(y_scaler.inverse_transform(mu_e.cpu().numpy()), dtype=torch.float32, device=device)
        y_te_orig = torch.tensor(y_te, dtype=torch.float32, device=device)
        var_s_orig = var_s * (y_std_scale ** 2)
        var_e_orig = var_e * (y_std_scale ** 2)

        rows['single_rmse'].append(evaluate_rmse_nll(y_te_orig, mu_s_orig, var_s_orig)[0])
        rows['ensemble_rmse'].append(evaluate_rmse_nll(y_te_orig, mu_e_orig, var_e_orig)[0])
        rows['single_nll'].append(evaluate_rmse_nll(y_te_orig, mu_s_orig, var_s_orig)[1])
        rows['ensemble_nll'].append(evaluate_rmse_nll(y_te_orig, mu_e_orig, var_e_orig)[1])

    def summarize(vals):
        return np.mean(vals), np.std(vals)

    print('\n=== Boston Housing (5-fold CV) ===')
    print(f"Paper Deep Ensembles (Table 1): RMSE ≈ 3.28±1.00, NLL ≈ 2.41±0.25")
    print(f"Single net (NLL):  RMSE = {summarize(rows['single_rmse'])[0]:.3f} ± {summarize(rows['single_rmse'])[1]:.3f}, "
          f"NLL = {summarize(rows['single_nll'])[0]:.3f} ± {summarize(rows['single_nll'])[1]:.3f}")
    print(f"Deep ensemble M={n_models}: RMSE = {summarize(rows['ensemble_rmse'])[0]:.3f} ± {summarize(rows['ensemble_rmse'])[1]:.3f}, "
          f"NLL = {summarize(rows['ensemble_nll'])[0]:.3f} ± {summarize(rows['ensemble_nll'])[1]:.3f}")

    labels = ['Single\nRMSE', 'Ensemble\nRMSE', 'Single\nNLL', 'Ensemble\nNLL']
    means = [np.mean(rows['single_rmse']), np.mean(rows['ensemble_rmse']),
             np.mean(rows['single_nll']), np.mean(rows['ensemble_nll'])]
    stds = [np.std(rows['single_rmse']), np.std(rows['ensemble_rmse']),
            np.std(rows['single_nll']), np.std(rows['ensemble_nll'])]

    fig, ax = plt.subplots(figsize=(6, 4))
    xpos = np.arange(len(labels))
    ax.bar(xpos, means, yerr=stds, capsize=5, color=['#4C72B0', '#55A868', '#C44E52', '#8172B2'])
    ax.set_xticks(xpos)
    ax.set_xticklabels(labels)
    ax.set_ylabel('Score (lower is better)')
    ax.set_title('Boston Housing — single vs. deep ensemble')
    plt.tight_layout()
    plt.savefig('boston_single_vs_ensemble.png', dpi=150, bbox_inches='tight')
    plt.show()
    return rows


boston_results = run_boston_cv()

## Uncertainty calibration (notes for the report)

- **In-distribution vs. OOD (toy):** Outside $[-4, 4]$, the NLL ensemble widens its $\pm 3\sigma$ band while the mean still tracks $x^3$ — epistemic uncertainty grows where data are sparse.
- **NLL vs. MSE:** Training with Eq. (1) yields heteroscedastic aleatoric noise; MSE ensembles only use disagreement across nets (often under-confident far from data).
- **Calibration check:** On Boston, compare **observed** vs. **predicted** coverage (e.g. fraction of test points inside 68%/95% predictive intervals). Well-calibrated models match nominal levels.
- **Paper reference (Table 1):** Deep Ensembles achieve competitive NLL on Boston; RMSE can be slightly worse because the objective is probabilistic, not purely point-error.